# EKF · Figuras

Trayectorias de la variante con perturbaciones cinéticas del notebook 08 para LAB019–021 y LAB016–018. En el flujo, la curva es el pronóstico de la ventana antes de incorporar la medición (\(\widehat y_{k|k-1}\)). En los estados, la curva muestra la estimación del EKF con las mediciones disponibles hasta ese instante (\(\widehat x_{k|k}\)).

Los puntos negros reúnen las muestras y el valor usado para inicializar cada estado. La inicial de etanol se tomó de la condición inicial del ensayo 08. En glucosa más fructosa, los rombos identifican la referencia por densidad. Se muestra el RMSE cuando hay una referencia directa para calcularlo.

In [ ]:
from pathlib import Path
import ast
import json
import os
import sys
from types import ModuleType

import matplotlib.pyplot as plt
import nbformat
import numpy as np
import pandas as pd
import scipy
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/wine_fermentation").is_dir())
EKF = ROOT / "notebooks/EKF"
HERE = EKF / "kalman_analysis"
prefix = Path(os.environ["CONDA_PREFIX"]).resolve()
assert prefix.name.lower() == "fermentation" and Path(sys.prefix).resolve() == prefix
print(f"Kernel {prefix.name} · SciPy {scipy.__version__}")

# Load the existing helpers as Python functions. The filter below uses the
# vectorized NumPy transition from EKF05 and does not require Numba.
core_path = HERE / "analysis_core.py"
tree = ast.parse(core_path.read_text(encoding="utf-8"))
tree.body = [node for node in tree.body
             if not (isinstance(node, ast.ImportFrom) and node.module == "numba")]
for node in tree.body:
    if isinstance(node, ast.FunctionDef):
        node.decorator_list = [
            d for d in node.decorator_list
            if not (isinstance(d, ast.Call) and isinstance(d.func, ast.Name) and d.func.id == "njit")
        ]
core = ModuleType("analysis_core")
core.__file__ = str(core_path)
exec(compile(tree, str(core_path), "exec"), vars(core))

# Reuse the six-lot input preparation from the four-source comparison.
reference = HERE / "04_gf_variable_solubility_fourth_noise_source.source.ipynb"
reference_nb = nbformat.read(reference, as_version=4)
preparation = next(cell.source for cell in reference_nb.cells
                   if cell.cell_type == "code" and "DATA,CHEM,AUDIT,_=core.prepare(ROOT)" in cell.source)
assert preparation.count("import analysis_core as core") == 1
preparation = preparation.replace("import analysis_core as core", "")
exec(compile(preparation.replace("display(AUDIT)", ""), str(reference), "exec"))

calibration = ROOT / "notebooks/diagnostics/zenteno_lab004_009_incertidumbre_exploratoria.ipynb"
saved_nb = nbformat.read(calibration, as_version=4)
saved = json.loads(saved_nb.cells[-1].outputs[-1].text)
rate_names = ["mu0", "betaG0", "betaF0"]
assert saved["orden_D_theta"] == rate_names
D_theta = np.array(saved["D_theta_Yef_fijo"], dtype=float)
np.testing.assert_array_equal(D_theta, np.diag(np.diag(D_theta)))
assert (np.diag(D_theta) > 0).all()
R = 0.00218
R_cell = next(cell.source for cell in reference_nb.cells
              if cell.cell_type == "code" and "R = 0.00218" in cell.source)
assert ast.literal_eval(ast.parse(R_cell).body[0].value) == R
theta = core.THETA
nominal = np.array([theta[name] for name in rate_names])
sigma_theta = np.sqrt(np.diag(D_theta))
SCALE = core.SCALE
scale_outer = np.outer(SCALE, SCALE)
KLA, GAIN = core.ARCHIVED_GAS
PCO2_BAR = 1.0


In [ ]:
# Reuse the equations from the vectorized G/F EKF. Supply one set of three
# rate coefficients per trajectory, with the other coefficients shared.
source05 = EKF / "05_zenteno_co2_variable_solubility.source.ipynb"
nb05 = nbformat.read(source05, as_version=4)
functions = {}
for cell in nb05.cells:
    if cell.cell_type == "code":
        for node in ast.parse(cell.source).body:
            if isinstance(node, ast.FunctionDef):
                functions[node.name] = ast.get_source_segment(cell.source, node)
for name in ("co2_saturation", "gas_exchange"):
    exec(compile(functions[name], str(source05), "exec"))

rates_source = functions["rates_ekf"]
for original, replacement in (
    ("def rates_ekf(w,t,T,S0):", "def rates_ekf(w,t,T,S0,rate_coefficients):"),
    ("theta['mu0']", "rate_coefficients[:, 0]"),
    ("theta['betaG0']", "rate_coefficients[:, 1]"),
    ("theta['betaF0']", "rate_coefficients[:, 2]"),
):
    assert rates_source.count(original) == 1
    rates_source = rates_source.replace(original, replacement)
exec(compile(rates_source, str(source05), "exec"))


def transition(points, rate_coefficients, start, end, data):
    """Integrate each trajectory with its own constant rates over this interval."""
    w = points * SCALE
    n = max(1, int(np.ceil((end - start) * 60 - 1e-7)))
    dt = (end - start) / n
    available = data["available"]
    temperatures = data["temperature"]
    for j in range(n):
        t = start + j * dt
        times = np.array([t, t + dt / 2, t + dt])
        indices = np.searchsorted(available, times + 1e-12, side="right") - 1
        if indices[0] < 0:
            raise ValueError("No hay temperatura causal al comienzo del intervalo")
        T0, Tm, T1 = temperatures[indices]
        S0 = data["S0"]
        k1 = rates_ekf(w, t, T0, S0, rate_coefficients)
        k2 = rates_ekf(w + dt * k1 / 2, t + dt / 2, Tm, S0, rate_coefficients)
        k3 = rates_ekf(w + dt * k2 / 2, t + dt / 2, Tm, S0, rate_coefficients)
        k4 = rates_ekf(w + dt * k3, t + dt, T1, S0, rate_coefficients)
        w = np.maximum(w + dt * (k1 + 2 * k2 + 2 * k3 + k4) / 6, 0.)
    return w / SCALE


def sensitivities(z, start, end, data, kinetic, relative_step=1e-4):
    state_step = 1e-5
    delta = np.eye(len(z)) * state_step
    lower = z - delta * (z >= state_step)[:, None]
    denominator = state_step * (1 + (z >= state_step))
    points = np.vstack([z, z + delta, lower])
    coefficients = np.tile(nominal, (len(points), 1))
    if kinetic:
        parameter_step = relative_step * nominal
        parameter_delta = np.diag(parameter_step)
        points = np.vstack([points, np.tile(z, (6, 1))])
        coefficients = np.vstack([coefficients, nominal + parameter_delta, nominal - parameter_delta])

    predicted = transition(points, coefficients, start, end, data)
    J = ((predicted[1:8] - predicted[8:15]) / denominator[:, None]).T
    G = ((predicted[15:18] - predicted[18:21]) / (2 * parameter_step[:, None])).T if kinetic else None
    return predicted[0], J, G


# Compare with the archived scalar RHS and transition in several regimes.
data = DATA["LAB019"]
for start, state in (
    (3., [0.4, .20, 90., 95., 15., .02, 0.]),
    (31., [1.8, .12, 65., 70., 36., .65, .01]),
    (75., [2.4, .03, 27., 43., 65., 1.0, .02]),
    (120., [.9, 0., 0., 12., 94., 1.2, .03]),
):
    w = np.asarray(state)
    z = w / SCALE
    temp = core.temperature_at(start, data["available"], data["temperature"])
    np.testing.assert_allclose(
        rates_ekf(w[None, :], start, temp, data["S0"], nominal[None, :])[0],
        core.rhs(w, temp, data["S0"], KLA, GAIN), atol=1e-13, rtol=1e-12,
    )
    end = start + 1 / 12
    zp, J, G = sensitivities(z, start, end, data, True)
    z_ref, J_ref = core.jacobian(z, start, end, data["available"], data["temperature"],
                                 data["S0"], KLA, GAIN, 1 / 60)
    np.testing.assert_allclose(zp, z_ref, atol=1e-13)
    np.testing.assert_allclose(J, J_ref, atol=1e-9)
    _, _, G_fine = sensitivities(z, start, end, data, True, relative_step=5e-5)
    np.testing.assert_allclose(G, G_fine, atol=5e-7, rtol=3e-3)
print("Verificadas la dinámica nominal, las sensibilidades y el paso de diferencias finitas.")


In [ ]:
flux_variances = np.square(core.BASE_SIGMA)

def replay(data, kinetic):
    grid = data["grid"]
    z = data["initial"] / SCALE
    P = data["P0"] / scale_outer
    states = np.empty((len(grid), len(z)))
    covariance = np.empty((len(grid), len(z), len(z)))
    diag = np.full((len(grid), 7), np.nan)
    injected = np.zeros_like(covariance)
    for i, end in enumerate(grid):
        if i:
            start = grid[i - 1]
            before = z
            z, J, G = sensitivities(z, start, end, data, kinetic)
            if kinetic:
                # G already differentiates the integrated transition.
                scaled_G = G * sigma_theta
                Q = scaled_G @ scaled_G.T
            else:
                T = core.temperature_at(start, data["available"], data["temperature"])
                L = core.noise_L(before * SCALE, T, data["S0"], KLA, GAIN)
                Q = (end - start) * ((L * flux_variances) @ L.T) / scale_outer
            injected[i] = Q * scale_outer
            P = J @ P @ J.T + Q
            P = (P + P.T) / 2

        z[1] += data["doses"][i] / SCALE[1]
        if end > 0 and abs(end / .5 - round(end / .5)) < 1e-8:
            predicted = z[6] / .5
            innovation = data["y"][i] - predicted
            S = 4 * P[6, 6] + R
            clips = 0
            if data["update"][i]:
                K = 2 * P[:, 6] / S
                updated = z + K * innovation
                clips = np.count_nonzero(updated[:6] < 0)
                B = np.eye(len(z))
                B[:, 6] -= 2 * K
                P = B @ P @ B.T + np.outer(K, K) * R
                P = (P + P.T) / 2
                z = np.maximum(updated, 0)
            diag[i] = [predicted, innovation, S, innovation**2 / S, clips,
                       data["update"][i], data["valid"][i]]
            z[6] = 0.
            P[6, :] = P[:, 6] = 0.
        states[i] = z * SCALE
        covariance[i] = P * scale_outer
    assert np.isfinite(states).all() and np.isfinite(covariance).all()
    assert np.linalg.eigvalsh(injected).min() > -1e-9
    np.testing.assert_array_equal(diag[:, 5] == 1, data["update"])
    return (states, covariance, diag), injected

records = []
results = {}
for lab in LABS:
    print(lab, flush=True)
    result, _ = replay(DATA[lab], kinetic=True)
    results[lab] = result
    scores, _, _ = core.diagnostics(DATA, CHEM, lab, result)
    records.append(scores[["LAB", "variable", "unit", "n", "RMSE"]].assign(Grupo=ROLE[lab]))
rmse_table = pd.concat(records, ignore_index=True)
rmse_table = rmse_table.sort_values(["Grupo", "LAB", "variable", "unit"])
display(rmse_table.set_index(["Grupo", "LAB", "variable", "unit"]).round({"RMSE": 5}))


In [ ]:
def state_series(frame, name):
    if name == "GF":
        return frame[:, 2] + frame[:, 3]
    return frame[:, core.STATES.index(name)]


def legend_inside(ax, error_label):
    handles, labels = ax.get_legend_handles_labels()
    entries = dict(zip(labels, handles))
    assert len(labels) == len(entries)
    legend = ax.legend(entries.values(), entries.keys(), loc="best",
                       fontsize=8, framealpha=.85, title=error_label, title_fontsize=8)
    if error_label:
        legend.get_title().set_ha("left")


def draw_panel(variable, title, unit, symbol):
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    for ax, lab in zip(axes.flat, LABS):
        d = DATA[lab]
        frame, _, diagnostics = results[lab]
        time = d["grid"]
        factor = 1000 if variable == "N" else 1
        if variable == "CO2":
            mask = d["valid"] & np.isfinite(diagnostics[:, 0])
            ax.plot(time[mask], diagnostics[mask, 0], c="#0072B2", lw=1.5, label=symbol)
            sensor = d["sensor"]
            ax.plot(sensor.t_h, sensor.measured, c="black", lw=1, label="Samples")
        else:
            ax.plot(time, factor * state_series(frame, variable),
                    c="#0072B2", lw=1.5, label=symbol)
            if variable in core.CHEM_COLS:
                column = core.CHEM_COLS[variable]
                samples = CHEM.loc[CHEM.lab.eq(lab) & CHEM.t_h.gt(0) &
                                   CHEM.t_h.le(d["horizon"])].dropna(subset=[column])
                if variable == "GF":
                    samples = samples.loc[~samples.GF_zero_substitution.astype(str).str.lower().eq("true")]
                initial = state_series(d["initial"][None, :], variable)[0]
                ax.scatter(np.r_[0., samples.t_h], factor * np.r_[initial, samples[column]],
                           c="black", s=23, label="Samples", zorder=5)
                if variable == "GF":
                    density = DENSITY.loc[DENSITY.lab.eq(lab) &
                                          DENSITY.t_h.between(0, d["horizon"])].dropna(subset=["density_g_l"])
                    if len(density):
                        from_density = 2.5 * (density.density_g_l - 1000)
                        ax.scatter(density.t_h, from_density, marker="D", facecolors="none",
                                   edgecolors="#D55E00", s=32, label="G+F from density", zorder=4)
                        ax.set_ylim(bottom=min(0., float(from_density.min()) * 1.15))
                        ax.axhline(0, c="gray", lw=.6)
        for i, pulse in enumerate(time[d["doses"] > 0]):
            ax.axvline(pulse, c="gray", ls=":", lw=.8, label="N pulse" if i == 0 else "_nolegend_")
        score = rmse_table.loc[rmse_table.LAB.eq(lab) & rmse_table.variable.eq(variable), "RMSE"]
        error_label = f"RMSE = {score.iloc[0]:.3f} {unit}" if len(score) else None
        if variable != "GF" or not ax.collections or ax.get_ylim()[0] > 0:
            ax.set_ylim(bottom=0.)
        ax.set(title=lab, xlabel="Time [h]", ylabel=f"{title} [{unit}]")
        ax.grid(alpha=.2)
        legend_inside(ax, error_label)
        assert sum(line.get_label() == symbol for line in ax.lines) == 1
        assert ax.get_legend().get_title().get_text() == (error_label or "")
    fig.suptitle(title, y=.99)
    fig.tight_layout(pad=.7, h_pad=1.2, w_pad=.9)
    assert not fig.legends
    plt.show()


for variable, title, unit, symbol in (
    ("CO2", "CO₂ flow", "g/(L h)", r"$\widehat{y}_{k|k-1}$"),
    ("X", "Biomass", "g/L", r"$\widehat{X}_{k|k}$"),
    ("N", "YAN", "mg/L", r"$\widehat{N}_{k|k}$"),
    ("G", "Glucose", "g/L", r"$\widehat{G}_{k|k}$"),
    ("F", "Fructose", "g/L", r"$\widehat{F}_{k|k}$"),
    ("GF", "Glucose + fructose", "g/L", r"$\widehat{G}_{k|k}+\widehat{F}_{k|k}$"),
    ("E", "Ethanol", "g/L", r"$\widehat{E}_{k|k}$"),
    ("Cd", "Dissolved CO₂", "g/L", r"$\widehat{C}_{d,k|k}$"),
):
    draw_panel(variable, title, unit, symbol)
